<a href="https://colab.research.google.com/github/Sanjarbek9706/computer_vision/blob/main/vegetable_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Vegetable Image Dataset bilan ishlash

Ushbu notebook Kaggle'dagi [Vegetable Image Dataset](https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset) (1-versiya) rasmlarini Colab'ga yuklaydi, `train`, `validation`, `test` bo'limlarini tekshiradi, `vegetable_manifest.csv` yaratadi va PyTorch `ImageFolder` uchun tayyorlaydi. Asl rasmlar GitHub'ga kiritilmaydi.

Manba: M Israk Ahmed va hamkorlar, *DCNN-Based Vegetable Image Classification Using Transfer Learning: A Comparative Study* (2021). Dataset litsenziyasi: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).


In [ ]:
%pip -q install "kagglehub>=1.0.3"


In [ ]:
from pathlib import Path
import csv
from collections import Counter

import kagglehub
from PIL import Image

DATASET_HANDLE = "misrakahmed/vegetable-image-dataset/versions/1"
download_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Kaggle fayllari:", download_dir)

def find_dataset_root(download_dir: Path) -> Path:
    """Ichida train, validation va test bor yagona papkani topadi."""
    candidates = [download_dir, *(p for p in download_dir.rglob('*') if p.is_dir())]
    matches = [p for p in candidates if all((p / split).is_dir() for split in ('train', 'validation', 'test'))]
    if len(matches) != 1:
        raise RuntimeError(f"Dataset ildizi aniq emas: {matches}. Kaggle fayl tuzilishini tekshiring.")
    return matches[0]

DATA_ROOT = find_dataset_root(download_dir)
print("Rasmlar papkasi:", DATA_ROOT)


In [ ]:
# Inglizcha papka nomlari model sinflari bo'lib qoladi; Uzbekcha nomlar interfeys uchun.
UZBEK_NAMES = {
    'Bean': 'loviya',
    'Bitter_Gourd': 'taxir qovoq',
    'Bottle_Gourd': 'idish qovoq',
    'Brinjal': 'baqlajon',
    'Broccoli': 'brokkoli',
    'Cabbage': 'karam',
    'Capsicum': "bulg'or qalampiri",
    'Carrot': 'sabzi',
    'Cauliflower': 'gulkaram',
    'Cucumber': 'bodring',
    'Papaya': 'papayya',
    'Potato': 'kartoshka',
    'Pumpkin': 'qovoq',
    'Radish': 'turp',
    'Tomato': 'pomidor',
}

SPLITS = ('train', 'validation', 'test')
split_classes = {
    split: {p.name for p in (DATA_ROOT / split).iterdir() if p.is_dir()}
    for split in SPLITS
}
assert all(split_classes[s] == split_classes['train'] for s in SPLITS), split_classes
assert split_classes['train'] == set(UZBEK_NAMES), {
    'kutilgan': sorted(UZBEK_NAMES), 'topilgan': sorted(split_classes['train'])
}
CLASSES = sorted(split_classes['train'])
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASSES)}
print('Sinflar:', CLASS_TO_IDX)


In [ ]:
# Har bir rasm uchun yo'l, bo'lim va sinf yozilgan manifest.
rows = []
for split in SPLITS:
    for class_name in CLASSES:
        class_dir = DATA_ROOT / split / class_name
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                rows.append({
                    'split': split,
                    'class_en': class_name,
                    'class_uz': UZBEK_NAMES[class_name],
                    'class_id': CLASS_TO_IDX[class_name],
                    'relative_path': image_path.relative_to(DATA_ROOT).as_posix(),
                })

if not rows:
    raise RuntimeError('Hech qanday rasm topilmadi')

MANIFEST = Path('/content/vegetable_manifest.csv')
with MANIFEST.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=('split', 'class_en', 'class_uz', 'class_id', 'relative_path'))
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row['split'] for row in rows)
print('Rasmlar soni:', dict(counts), '| jami:', len(rows))
print('Manifest:', MANIFEST)
assert all(counts[s] > 0 for s in SPLITS)


In [ ]:
# Dataset sifatini tekshirish: har bo'limdagi har sinfdan 3 tadan.
sample_paths = []
for split in SPLITS:
    for class_name in CLASSES:
        sample_paths.extend(sorted((DATA_ROOT / split / class_name).glob('*.jpg'))[:3])

for image_path in sample_paths:
    with Image.open(image_path) as image:
        image.verify()
print(f'{len(sample_paths)} ta namunaviy rasm tekshirildi')


In [ ]:
# Modelga berish uchun ImageFolder va DataLoader.
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=transform)
validation_ds = datasets.ImageFolder(DATA_ROOT / 'validation', transform=transform)
test_ds = datasets.ImageFolder(DATA_ROOT / 'test', transform=transform)
assert train_ds.class_to_idx == validation_ds.class_to_idx == test_ds.class_to_idx == CLASS_TO_IDX

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
images, labels = next(iter(train_loader))
print('Birinchi batch:', tuple(images.shape), tuple(labels.shape))


## Loyihaga ulash

`CLASSES`, `CLASS_TO_IDX`, `train_loader`, `validation_loader` va `test_loader` model notebookida ishlatiladi. Model bitta suratdagi **bitta asosiy sabzavotni** taniydi. Bir suratda bir nechta mahsulotni topish uchun keyinroq object detection kerak bo'ladi.

Bu datasetda piyoz, sarimsoq, guruch va go'sht sinflari yo'q. Retsept tavsiyasi uchun ularni alohida dataset bilan qo'shish yoki foydalanuvchidan qo'lda kiritishni so'rash kerak.
